# Reviews

Address comments.

In [1]:
import ee
import geemap
from utils import *
initialize()

config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder


Successfully saved authorization token.


In [4]:
# there are objects defined in scripts 1 - 8 that will be used here. This requires running them in this script:

import nbimporter # lets you import notebooks like regular modules

%run 3_grids.ipynb # run full script here
%run 6_mature.ipynb # run full script here
%run 7_write_csv.ipynb # run full script here
%run objects.ipynb

In [3]:
sd = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').first().select("SD").rename("sd")

## GEDI - mean biomass per 10km grid cell

Aggregating GEDI L4A into 10km pixels doesn't work as fast and as well with reduceResolution + reproject (as done on 6_mature.ipynb). That process ends up running out of computational power.

In order to avoid this, it is necessary to get the mean per 10km grid cell over many tiles.


In [10]:
def quality_mask(image):
    image = image.updateMask(image.select('l4_quality_flag').eq(1)) \
              .updateMask(image.select('degrade_flag').eq(0))
    relative_se = image.select('agbd_se').divide(image.select('agbd'))
    return image.updateMask(relative_se.lte(0.5))

GEDI = (ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY')
        .filterDate('2015-01-01', '2020-12-31')
             .map(quality_mask)
             .select(['agbd']))

GEDI_projection = GEDI.first().projection()

GEDI = GEDI.mean().toInt16().reproject(GEDI_projection).rename('GEDI_biomass')

GEDI_mature = GEDI.updateMask(mature_mask.And(amazon_mask)).rename("GEDI_mature_biomass")

In [ ]:
# Aggregate the high-resolution pixels into the 10 km grid
mature_biomass_10k = GEDI_mature.reduceResolution(
    reducer = ee.Reducer.mean(),
    maxPixels = 1024,
    bestEffort = True # Use all pixels that can fit in the larger pixel
)

export_image(mature_biomass_10k, "mature_biomass_gedi_10k", scale = 10000, region = roi)

In [ ]:
# checking if this code is even needed - maybe we won't run out of memory anymore if I am masking and not clipping

# grid = amazon.geometry().coveringGrid('EPSG:4326', 200000)  # 200km tiles

# # Convert to a list so we can iterate
# grid_list = grid.toList(grid.size())
# n_tiles = grid.size().getInfo()
# print(f"Number of tiles: {n_tiles}")

# for i in range(n_tiles):
#     tile = ee.Feature(grid_list.get(i)).geometry()
    
#     # Aggregate within this tile only
#     tile_image = GEDI_mature \
#         .setDefaultProjection(crs='EPSG:4326', scale = 25) \
#         .reduceResolution(
#             reducer=ee.Reducer.mean(),
#             maxPixels=1024,
#             bestEffort=True
#         ) \
#         .reproject(crs='EPSG:4326', scale = 10000) \
#         .clip(tile)
    
#     task = ee.batch.Export.image.toAsset(
#         image=tile_image,
#         description=f"GEDI_mature_biomass_10k_tile_{i}",
#         assetId=f"{data_folder}/GEDI_mature/GEDI_mature_biomass_10k_tile_{i}",
#         region=tile,
#         crs='EPSG:4326',
#         scale=10000,
#         maxPixels=1e13
#     )
#     # task.start()
#     print(f"Started tile {i}/{n_tiles}")




Number of tiles: 148
Started tile 0/148
Started tile 1/148
Started tile 2/148
Started tile 3/148
Started tile 4/148
Started tile 5/148
Started tile 6/148
Started tile 7/148
Started tile 8/148
Started tile 9/148
Started tile 10/148
Started tile 11/148
Started tile 12/148
Started tile 13/148
Started tile 14/148
Started tile 15/148
Started tile 16/148
Started tile 17/148
Started tile 18/148
Started tile 19/148
Started tile 20/148
Started tile 21/148
Started tile 22/148
Started tile 23/148
Started tile 24/148
Started tile 25/148
Started tile 26/148
Started tile 27/148
Started tile 28/148
Started tile 29/148
Started tile 30/148
Started tile 31/148
Started tile 32/148
Started tile 33/148
Started tile 34/148
Started tile 35/148
Started tile 36/148
Started tile 37/148
Started tile 38/148
Started tile 39/148
Started tile 40/148
Started tile 41/148
Started tile 42/148
Started tile 43/148
Started tile 44/148
Started tile 45/148
Started tile 46/148
Started tile 47/148
Started tile 48/148
Started t

### GEDI nearest neighbor
After the GEDI data is aggregated to 10km resolution, we use the same method from 6_mature to get the nearest mature biomass for the gaps in the image (areas where there was no GEDI read on mature forests in the 10km grid cell pixel)

In [ ]:
all_images = import_folder_features(f"{data_folder}/GEDI_mature", asset_type='image')
mature_biomass_10k = ee.ImageCollection(all_images).mosaic()

features_secondary = ee.FeatureCollection(f"{data_folder}/grid_10k_amazon_secondary_edge_removed")

# obtain_nearest_mature_neighbor(mature_biomass_10k, features_secondary, "_GEDI")

### Grid sampling - GEDI

Use the same sampling method to obtain one pixel of secondary forest that coincides with a GEDI footprint per 10km2

Objects imported from 3_grids.ipynb


In [ ]:
GEDI_reproj = GEDI.reproject(age_edge_removed.projection()).rename("GEDI_biomass")

secondary_GEDI = age_edge_removed.updateMask(GEDI_reproj).selfMask()

# create_grid(export_image, region_name = "amazon", cell_size = 10000, file_name = "secondary_edge_removed_gedi")

In [22]:
grid_gedi = ee.FeatureCollection(f"{data_folder}/grid_10k_amazon_secondary_edge_removed_gedi")

nearest_mature_GEDI = ee.Image(f"{data_folder}/nearest_mature_GEDI").rename("nearest_mature_GEDI")

unified_data_secondary = ee.Image.cat([unified_data, age, GEDI_reproj, nearest_mature_GEDI, reduce_reproject(biomass, ee.Reducer.mean()).rename("ESA_biomass")])

export_csv("secondary_GEDI_ESA", unified_data_secondary, 10, n_chunks = 30, grid = grid_gedi)

## Same-age patches

Keep only patches of the same age and greater than 1ha

Do not exclude edge pixels. Selecting exclusively based on area and age.

In [ ]:
grid = amazon.geometry().coveringGrid('EPSG:4326', 100000)

grid_list = grid.toList(grid.size())
n = grid.size().getInfo()

for i in range(n):
    cell = ee.Feature(grid_list.get(i))

    vectors = age.reduceToVectors(
        geometry=cell.geometry(),
        geometryType='polygon',
        scale=30,
        eightConnected=True,
        maxPixels=1e12,
        labelProperty='age',
        tileScale = 16
    )

    vectors = vectors.map(
        lambda f: f.set({
            'area_m2': f.geometry().area(maxError=1),
            'tile_id': i + 1
        })
    ).filter(ee.Filter.gt('area_m2', 10000)).select(['age', 'tile_id'])

    task = ee.batch.Export.table.toAsset(
        collection=vectors,
        description=f"secondary_age_vectors_{i+1:03d}",
        assetId=f"{data_folder}/secondary_polygons/secondary_age_vectors_{i+1:03d}"
    )
    # task.start()

#309 needs to be run again

### Export age and biomass for ESA CCI for the same-age patches

In [ ]:
# for each collection, we will make a new collection by sampling one random pixel of age within it and returning it as a point feature.

def one_point_per_poly(f):
    geom = f.geometry()
    pt = ee.FeatureCollection.randomPoints(
        region = geom,
        points = 1,
        seed = ee.Number(1),   # or derive a deterministic seed if needed
        maxError = 1
    ).first()
    return ee.Feature(pt).copyProperties(f)

asset_list = ee.data.listAssets({'parent': f"{data_folder}/secondary_polygons"})['assets']

feature_ids = [a['name'] for a in asset_list]                    

for asset_id in feature_ids:
    polygons_fc = ee.FeatureCollection(asset_id)

    points_fc = ee.FeatureCollection(polygons_fc.map(one_point_per_poly))

    task = ee.batch.Export.table.toAsset(
        collection = points_fc,
        description = f"secondary_age_points_{asset_id[-3:]}",
        assetId = f"{data_folder}/secondary_points/secondary_age_vectors_{asset_id[-3:]}"
    )
    # task.start()
